In [51]:
# Load the dataset from the uploaded file
import pandas as pd

# File path
file_path = 'SeoulBikeData.csv'

# Read the data
df = pd.read_csv(file_path)

# Remove the 'Date' column and check for null values
df.drop(columns=['Date'], inplace=True)
null_values = df.isnull().sum()

# Display the null values count
null_values


In [53]:
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=['Seasons', 'Holiday', 'Functioning Day'], drop_first=True)

# Separate predictors (X) and target variable (y)
X = df_encoded.drop(columns=['Rented Bike Count'])
y = df_encoded['Rented Bike Count']

# Display the first few rows of predictors and target variable
X.head(), y.head()


In [60]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split standardized data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Build the neural network
nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)  # Output layer for regression
])

# Compile the model
nn_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# Train the model
history = nn_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

# Evaluate the model
nn_loss, nn_mae = nn_model.evaluate(X_test, y_test, verbose=0)

# Predict on the test set
nn_predictions = nn_model.predict(X_test)

# Calculate RMSE and R²
nn_rmse = mean_squared_error(y_test, nn_predictions, squared=False)
nn_r2 = r2_score(y_test, nn_predictions)

{
    "Neural Network RMSE": nn_rmse,
    "Neural Network R^2": nn_r2
}


In [59]:
def predict_bike_count(model, hour, temperature, humidity, wind_speed, visibility, dew_point_temp, solar_radiation, rainfall, snowfall, season, holiday, functioning_day):
    # Map categorical inputs to the one-hot encoded format
    season_mapping = {'Spring': [1, 0, 0], 'Summer': [0, 1, 0], 'Winter': [0, 0, 1], 'Autumn': [0, 0, 0]}
    holiday_mapping = {'No Holiday': [1], 'Holiday': [0]}
    functioning_day_mapping = {'Yes': [1], 'No': [0]}

    # Prepare the input data
    input_data = [
        hour,
        temperature,
        humidity,
        wind_speed,
        visibility,
        dew_point_temp,
        solar_radiation,
        rainfall,
        snowfall
    ] + season_mapping.get(season, [0, 0, 0]) + holiday_mapping.get(holiday, [1]) + functioning_day_mapping.get(functioning_day, [1])

    # Convert to DataFrame format (matching training data structure)
    input_df = pd.DataFrame([input_data], columns=X.columns)

    # Make prediction
    prediction = model.predict(input_df)

    return int(prediction[0])

# Example usage
predicted_count = predict_bike_count(
    model=rf_model,
    hour=5,
    temperature=-13.0,
    humidity=1,
    wind_speed=1.2,
    visibility=1996,
    dew_point_temp=-21.2,
    solar_radiation=0.65,
    rainfall=0.0,
    snowfall=0.0,
    season='winter',
    holiday='No Holiday',
    functioning_day='no'
)
print(f"Predicted bike count: {predicted_count}")
